# Focus Area 4 — Agro-Ecological Zoning with PyAEZ

This notebook bridges the climate monitoring work from Focus Areas 1–3 into agricultural interpretation using **PyAEZ v2.2** (FAO / AIT).

**What it computes:**
- **Thermal Climate Classification** — which climate zones exist in the country
- **Length of Growing Period (LGP)** — how many days per year support crop growth
- **Agro-Ecological Zone (AEZ) map** — spatial distribution of zones
- **Crop suitability summary** — which crops are viable and where

**Inputs drawn from Shared Drive (cached by FA1–FA3):**
- ERA5 monthly temperature (tmin, tavg, tmax) — from FA3
- Best-scoring satellite precipitation (CHIRPS/TAMSAT/ERA5/IMERG) — from FA2
- PET (Hargreaves) — recomputed here from temperature

---

In [ ]:
# @title ### 0) Configuration {"display-mode":"form"}
# @markdown Set the country and which satellite product to use for precipitation.
# @markdown The precipitation product should be the one that scored highest in Focus Area 2.

import os
import numpy as np
from datetime import date

# ── COUNTRY ───────────────────────────────────────────────────────────────────
COUNTRY = 'Ethiopia'    # 'Ethiopia' | 'Tanzania' | 'Eritrea' | 'Djibouti'

COUNTRY_BBOX = {
    'Ethiopia':  [33.0,  3.0, 48.0, 15.0],
    'Tanzania':  [29.0, -12.0, 41.0, -1.0],
    'Eritrea':   [36.5,  12.5, 43.5, 18.0],
    'Djibouti':  [41.5,  10.9, 43.5, 12.7],
}
xmin, ymin, xmax, ymax = COUNTRY_BBOX[COUNTRY]

# ── PRECIPITATION SOURCE ──────────────────────────────────────────────────────
# Set to whichever product scored highest in FA2 for this country.
# Options: 'CHIRPS' | 'TAMSAT' | 'ERA5' | 'IMERG'
PRECIP_SOURCE = 'CHIRPS'

# ── DATE RANGE FOR LONG-TERM ANALYSIS ────────────────────────────────────────
# Uses the long-term ERA5 from FA3 (monthly 2020–2024 by default).
LT_START = 2020
LT_END   = 2024

# ── LOCAL DIRS ────────────────────────────────────────────────────────────────
LOCAL_CACHE_DIR = f'./Datasets/{COUNTRY}'
RESULTS_DIR     = f'./outputs/FA4_AEZ_{COUNTRY}_{date.today().isoformat()}'
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,     exist_ok=True)

print(f'✅ CONFIG set.')
print(f'   Country         : {COUNTRY}')
print(f'   Bbox            : [{xmin}, {ymin}, {xmax}, {ymax}]')
print(f'   Precip source   : {PRECIP_SOURCE}')
print(f'   Long-term period: {LT_START}–{LT_END}')
print(f'   Outputs         : {RESULTS_DIR}')

In [ ]:
# @title ### 1) GEE Authentication {"display-mode":"form"}
# @markdown Same service account used across all Focus Area notebooks.

# @title ### 1) GEE Authentication {"display-mode":"form"}

import ee
import json
import gdown

# ── Service account key — downloaded from Shared Drive at runtime ──────────────
# File ID from: https://drive.google.com/file/d/181IKB3sJ3iUn6ZOZbg50htgH2JKcxFkT
SA_KEY_FILE_ID = '181IKB3sJ3iUn6ZOZbg50htgH2JKcxFkT'
SA_KEY_PATH    = 'service_account_key.json'

print('🔑 Downloading service account key...')
gdown.download(
    f'https://drive.google.com/uc?id={SA_KEY_FILE_ID}',
    SA_KEY_PATH,
    quiet=True
)

if not os.path.exists(SA_KEY_PATH) or os.path.getsize(SA_KEY_PATH) == 0:
    raise RuntimeError(
        '❌ Key file download failed.\n'
        '   → Make sure the file is shared as "Anyone with the link" in Google Drive.'
    )

with open(SA_KEY_PATH) as f:
    _key = json.load(f)

credentials = ee.ServiceAccountCredentials(
    email=_key['client_email'],
    key_file=SA_KEY_PATH
)
ee.Initialize(credentials)
print(f'✅ GEE authenticated as: {_key["client_email"]}')

In [ ]:
# @title ### 2) Install dependencies & mount Shared Drive {"display-mode":"form"}
print('Installing dependencies...')

# ── Step 1: Install system GDAL (must come before any pip install that needs it)
!apt-get install -y libgdal-dev gdal-bin > /dev/null 2>&1

# ── Step 2: Install GDAL Python bindings pinned to the system version
import subprocess
_gdal_ver = subprocess.check_output(['gdal-config','--version']).decode().strip()
print(f'   System GDAL version: {_gdal_ver}')
!pip install gdal[numpy]=={_gdal_ver} -q

# ── Step 3: Now install PyAEZ (gdal is available so the wheel can build)
!pip install pyaez==2.2 numba scipy -q
!pip install xarray rioxarray rasterio cartopy -q

print('✅ Dependencies installed.')

# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os, io, json, shutil
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from datetime import date
from google.colab import drive

# ── PyAEZ — try multiple import paths since internal structure varies ─────────
# PyAEZ v2.2 internal files: ClimateRegime.py, UtilityCalculations.py
# They live inside the pyaez package folder but __init__.py may not expose them
PYAEZ_AVAILABLE = False
ClimateRegime      = None
UtilityCalculations = None

try:
    # Primary path (v2.2 recommended)
    from pyaez import ClimateRegime as _CR
    ClimateRegime = _CR
    print('✅ PyAEZ ClimateRegime imported.')
    PYAEZ_AVAILABLE = True
except ImportError as e:
    print(f'⚠️  ClimateRegime import failed: {e}')

try:
    from pyaez import UtilityCalculations as _UC
    UtilityCalculations = _UC
    print('✅ PyAEZ UtilityCalculations imported.')
except ImportError:
    # Try the direct file import — works when __init__.py doesn't export it
    try:
        import importlib, pyaez as _pyaez_pkg, os as _os
        _pkg_dir = _os.path.dirname(_pyaez_pkg.__file__)
        _spec = importlib.util.spec_from_file_location(
            'UtilityCalculations',
            _os.path.join(_pkg_dir, 'UtilityCalculations.py')
        )
        if _spec:
            UtilityCalculations = importlib.util.module_from_spec(_spec)
            _spec.loader.exec_module(UtilityCalculations)
            print('✅ PyAEZ UtilityCalculations imported via direct file path.')
        else:
            print('⚠️  UtilityCalculations.py not found in package — Ra will use approximation.')
    except Exception as e2:
        print(f'⚠️  UtilityCalculations not available ({e2}) — Ra will use approximation.')

if not PYAEZ_AVAILABLE:
    print('⚠️  PyAEZ not available — analysis will use equivalent manual FAO calculations.')

# ── MOUNT SHARED DRIVE ────────────────────────────────────────────────────────
print('\n📂 Mounting Google Drive...')
drive.mount('/content/drive', force_remount=False)

SHARED_DRIVE_ROOT = '/content/drive/Shareddrives/NOAA-workshop2'
DATASETS_ROOT     = os.path.join(SHARED_DRIVE_ROOT, 'Datasets')
COUNTRY_DRIVE_DIR = os.path.join(DATASETS_ROOT, COUNTRY)

if not os.path.exists(SHARED_DRIVE_ROOT):
    print('❌ Shared Drive not found. Check your access to NOAA-workshop2.')
    DRIVE_CACHE_AVAILABLE = False
else:
    os.makedirs(COUNTRY_DRIVE_DIR, exist_ok=True)
    DRIVE_CACHE_AVAILABLE = True
    print(f'✅ Shared Drive ready: {COUNTRY_DRIVE_DIR}')

def try_load_from_drive(local_path, filename):
    if not DRIVE_CACHE_AVAILABLE: return False
    src = os.path.join(COUNTRY_DRIVE_DIR, filename)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(local_path) or '.', exist_ok=True)
        shutil.copy(src, local_path)
        print(f'   ✅ Loaded from Shared Drive: {filename}')
        return True
    return False

def cache_to_drive(local_path, filename):
    if not DRIVE_CACHE_AVAILABLE: return
    dest = os.path.join(COUNTRY_DRIVE_DIR, filename)
    try:
        shutil.copy(local_path, dest)
        print(f'   ☁️  Cached to Shared Drive: {filename}')
    except Exception as e:
        print(f'   ⚠️  Cache failed: {e}')

In [ ]:
# @title ### 3) Load climate inputs from Shared Drive {"display-mode":"form"}
# @markdown Reads the long-term ERA5 temperature (produced by FA3 Step 7)
# @markdown and the best satellite precipitation (produced by FA2) from the Shared Drive.
# @markdown Both are kept as SPATIAL grids for mapping — NOT collapsed to country mean.

import ee

# ── HELPER: subset xarray to country bbox ─────────────────────────────────────
def subset_to_bbox(ds, xmin, xmax, ymin, ymax):
    lat_dim = next((c for c in ds.coords if 'lat' in c.lower() or c == 'y'), None)
    lon_dim = next((c for c in ds.coords if 'lon' in c.lower() or c == 'x'), None)
    if lat_dim is None or lon_dim is None:
        return ds
    lv = ds[lat_dim].values
    ls = slice(ymax, ymin) if lv[0] > lv[-1] else slice(ymin, ymax)
    return ds.sel({lat_dim: ls, lon_dim: slice(xmin, xmax)})

# ── A) LONG-TERM ERA5 TEMPERATURE — kept as SPATIAL grid ─────────────────────
_lt_filename = f'era5_temperature_monthly_{LT_START}_{LT_END}_{COUNTRY}.nc'
_lt_local    = os.path.join(LOCAL_CACHE_DIR, _lt_filename)
era5_lt      = None

print(f'\n🔍 Loading ERA5 temperature: {_lt_filename}')

if os.path.exists(_lt_local):
    era5_lt = xr.open_dataset(_lt_local)
    print('   ✅ Found locally.')
elif try_load_from_drive(_lt_local, _lt_filename):
    era5_lt = xr.open_dataset(_lt_local)

# If not found, pull from GEE as a spatial grid (getDownloadURL → NetCDF)
if era5_lt is None:
    print('   ⚠️  Not found — pulling spatial grid from GEE...')
    try:
        import requests, zipfile, io as _io
        _region = ee.Geometry.BBox(xmin, ymin, xmax, ymax)
        _col = (
            ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
            .filterDate(f'{LT_START}-01-01', f'{LT_END}-12-31')
            .select('temperature_2m')
        )
        # Export as multi-band GeoTIFF and convert to xarray
        _imgs = _col.toList(_col.size())
        _n    = _col.size().getInfo()
        print(f'   Pulling {_n} monthly images...')

        bands_list = []
        times_list = []
        for i in range(_n):
            img   = ee.Image(_imgs.get(i))
            t_str = img.date().format('YYYY-MM-dd').getInfo()
            arr   = img.sampleRectangle(region=_region, defaultValue=0)
            vals  = np.array(arr.get('temperature_2m').getInfo(), dtype=float)
            vals  = np.where(vals > 150, vals - 273.15, vals)  # K → C
            bands_list.append(vals)
            times_list.append(pd.Timestamp(t_str))
            if (i+1) % 12 == 0:
                print(f'   ... {i+1}/{_n} months pulled')

        # Stack into (time, lat, lon)
        _stack  = np.stack(bands_list, axis=0)
        _nrows, _ncols = bands_list[0].shape
        _lats   = np.linspace(ymax, ymin, _nrows)
        _lons   = np.linspace(xmin, xmax, _ncols)
        era5_lt = xr.Dataset(
            {'temperature_2m': (['time','lat','lon'], _stack)},
            coords={'time': times_list, 'lat': _lats, 'lon': _lons}
        )
        era5_lt.to_netcdf(_lt_local)
        cache_to_drive(_lt_local, _lt_filename)
        print(f'   ✅ Spatial grid pulled and cached: {_stack.shape}')
    except Exception as e:
        print(f'   ❌ GEE pull failed: {e}')
        era5_lt = None

if era5_lt is not None:
    era5_lt = subset_to_bbox(era5_lt, xmin, xmax, ymin, ymax)
    _var0   = list(era5_lt.data_vars)[0]
    _da     = era5_lt[_var0]
    print(f'   Grid shape : {_da.shape}  (time, lat, lon)')
    print(f'   Time range : {str(_da.time.values[0])[:10]} → {str(_da.time.values[-1])[:10]}')

# ── B) SATELLITE PRECIPITATION — kept as SPATIAL grid ────────────────────────
_precip_filename = None
precip_ds        = None

print(f'\n🔍 Searching for {PRECIP_SOURCE} precipitation...')

if DRIVE_CACHE_AVAILABLE:
    _matching = [
        f for f in os.listdir(COUNTRY_DRIVE_DIR)
        if PRECIP_SOURCE.upper() in f.upper() and f.endswith('.nc')
    ]
    if _matching:
        _precip_filename = _matching[0]
        _precip_local    = os.path.join(LOCAL_CACHE_DIR, _precip_filename)
        if not os.path.exists(_precip_local):
            shutil.copy(os.path.join(COUNTRY_DRIVE_DIR, _precip_filename), _precip_local)
        precip_ds = xr.open_dataset(_precip_local)
        precip_ds = subset_to_bbox(precip_ds, xmin, xmax, ymin, ymax)
        _pvar = list(precip_ds.data_vars)[0]
        print(f'   ✅ Found: {_precip_filename}')
        print(f'   Grid shape: {precip_ds[_pvar].shape}')
    else:
        print(f'   ⚠️  No {PRECIP_SOURCE} NetCDF found in Shared Drive/{COUNTRY}/')
        print(f'   Available files: {os.listdir(COUNTRY_DRIVE_DIR)}')

print('\n📊 Input summary:')
print(f'   ERA5 temperature : {"✅ spatial grid" if era5_lt is not None else "❌ missing"}')
print(f'   Precipitation    : {"✅ " + str(_precip_filename) if precip_ds is not None else "❌ missing"}')


In [ ]:
# @title ### 4) Prepare monthly climate arrays for PyAEZ {"display-mode":"form"}
# @markdown Produces TWO versions of each climate variable:
# @markdown  • 3D spatial grids  (nrows, ncols, 12) → for maps and PyAEZ spatial run
# @markdown  • 1D country means  (12,)              → for time-series charts

MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# ── HELPER: xarray → monthly climatology, returns (12, lat, lon) ──────────────
def to_monthly_climatology(ds, method='mean'):
    """
    Resample to monthly, then average across years to get a 12-month climatology.
    Returns numpy array shape (12, nrows, ncols) or (12,) if no spatial dims.
    """
    var0 = list(ds.data_vars)[0]
    da   = ds[var0]
    if method == 'mean':
        monthly = da.resample(time='1ME').mean()
    else:
        monthly = da.resample(time='1ME').sum()
    climatology = monthly.groupby('time.month').mean('time')
    return climatology   # xarray DataArray with month as first dim

# ── TEMPERATURE: spatial climatology ─────────────────────────────────────────
print('Preparing temperature arrays...')

if era5_lt is not None:
    _var0 = list(era5_lt.data_vars)[0]
    _da   = era5_lt[_var0]

    # Convert K→C if needed
    if float(_da.mean()) > 100:
        _da = _da - 273.15

    # Monthly climatology — shape (12, nrows, ncols) or (12,)
    _monthly   = _da.resample(time='1ME').mean()
    _clim_da   = _monthly.groupby('time.month').mean('time')
    tavg_clim  = _clim_da.values   # (12, nrows, ncols) or (12,)

    # Pull lat/lon coords for mapping
    _lat_dim = next((c for c in era5_lt.coords if 'lat' in c.lower()), None)
    _lon_dim = next((c for c in era5_lt.coords if 'lon' in c.lower()), None)
    if _lat_dim and _lon_dim:
        map_lats = era5_lt[_lat_dim].values
        map_lons = era5_lt[_lon_dim].values
        HAS_SPATIAL = tavg_clim.ndim == 3
    else:
        HAS_SPATIAL = False

    # 1D country mean for charts
    if HAS_SPATIAL:
        tavg_monthly_1d = np.nanmean(tavg_clim, axis=(1, 2))
    else:
        tavg_monthly_1d = tavg_clim.flatten()[:12]
        HAS_SPATIAL     = False

    # Estimate tmin/tmax from tavg (±5°C diurnal range — East Africa typical)
    DIURNAL_RANGE = 10.0
    if HAS_SPATIAL:
        tmin_clim = tavg_clim - (DIURNAL_RANGE / 2)
        tmax_clim = tavg_clim + (DIURNAL_RANGE / 2)
    tmin_monthly_1d = tavg_monthly_1d - (DIURNAL_RANGE / 2)
    tmax_monthly_1d = tavg_monthly_1d + (DIURNAL_RANGE / 2)

    print(f'   ✅ Spatial shape: {tavg_clim.shape}  |  HAS_SPATIAL={HAS_SPATIAL}')
    print(f'   ✅ Monthly Tavg (country mean °C): {np.round(tavg_monthly_1d, 1)}')
else:
    print('⚠️  ERA5 temperature not available — using East Africa typical values.')
    tavg_monthly_1d = np.array([22,23,24,24,22,19,18,19,21,22,22,21], dtype=float)
    tmin_monthly_1d = tavg_monthly_1d - 5.0
    tmax_monthly_1d = tavg_monthly_1d + 5.0
    tavg_clim = tmin_clim = tmax_clim = None
    HAS_SPATIAL = False
    map_lats = map_lons = None

# ── PRECIPITATION: spatial climatology ────────────────────────────────────────
print('\nPreparing precipitation arrays...')

precip_clim       = None
precip_monthly_1d = None

if precip_ds is not None:
    _pvar = list(precip_ds.data_vars)[0]
    _pda  = precip_ds[_pvar]

    # Monthly sum climatology
    _p_monthly = _pda.resample(time='1ME').sum()
    _p_clim_da = _p_monthly.groupby('time.month').mean('time')
    precip_clim_raw = _p_clim_da.values   # (n_months, ...) or (12, lat, lon)

    # Handle partial-year data (e.g. only 6 months available)
    n_months = precip_clim_raw.shape[0]

    if n_months < 12:
        _t0 = str(precip_ds.time.values[0])[:10]
        _t1 = str(precip_ds.time.values[-1])[:10]
        print(f'   ℹ️  Precipitation covers {n_months} months ({_t0} → {_t1})')

        # Detect which calendar months are present
        _ptimes     = pd.DatetimeIndex([pd.Timestamp(str(t)[:10]) for t in precip_ds.time.values])
        _p_monthly_t= _pda.resample(time='1ME').sum()
        _p_months   = pd.DatetimeIndex([pd.Timestamp(str(t)[:10]) for t in _p_monthly_t.time.values])
        _p_clim_t   = _p_monthly_t.groupby('time.month').mean('time')
        _known_months_idx = [int(m)-1 for m in _p_clim_t['month'].values]

        # Build full 12-month array
        if precip_clim_raw.ndim == 1:
            precip_clim_full = np.full(12, np.nan)
            for ki, val in zip(_known_months_idx, precip_clim_raw):
                precip_clim_full[ki] = val
        else:
            _shape = (12,) + precip_clim_raw.shape[1:]
            precip_clim_full = np.full(_shape, np.nan)
            for ki, i in zip(_known_months_idx, range(len(_known_months_idx))):
                precip_clim_full[ki] = precip_clim_raw[i]

        # Fill missing months with mean of known months
        _known_mean = np.nanmean(precip_clim_full)
        _missing    = [i for i in range(12) if np.all(np.isnan(precip_clim_full[i]))]
        for mi in _missing:
            precip_clim_full[mi] = _known_mean
        precip_clim = precip_clim_full
        print(f'   ✅ Known months: {[i+1 for i in _known_months_idx]}')
        print(f'   ⚠️  Estimated months: {[i+1 for i in _missing]} (filled with mean = {_known_mean:.1f} mm)')
    else:
        precip_clim = precip_clim_raw

    # 1D country mean
    if precip_clim.ndim > 1:
        precip_monthly_1d = np.nanmean(precip_clim, axis=tuple(range(1, precip_clim.ndim)))
        HAS_SPATIAL_PRECIP = True
        # Update spatial coords from precip if ERA5 didn't have them
        if map_lats is None:
            _plat = next((c for c in precip_ds.coords if 'lat' in c.lower()), None)
            _plon = next((c for c in precip_ds.coords if 'lon' in c.lower()), None)
            if _plat and _plon:
                map_lats = precip_ds[_plat].values
                map_lons = precip_ds[_plon].values
    else:
        precip_monthly_1d = precip_clim
        HAS_SPATIAL_PRECIP = False
    print(f'   ✅ Monthly precip (country mean mm): {np.round(precip_monthly_1d, 1)}')
else:
    print('⚠️  Precipitation not available — using East Africa typical values.')
    precip_monthly_1d = np.array([40,50,90,120,100,30,15,20,50,90,80,45], dtype=float)
    precip_clim = None
    HAS_SPATIAL_PRECIP = False

# ── HARGREAVES PET ────────────────────────────────────────────────────────────
print('\nComputing monthly PET (Hargreaves)...')

lat_c = (ymin + ymax) / 2

if PYAEZ_AVAILABLE and UtilityCalculations is not None:
    try:
        UC = UtilityCalculations.UtilityCalculations()
        Ra_monthly = np.array([float(UC.calcRa(lat_c, m)) for m in range(1, 13)])
        print(f'   ✅ Ra from PyAEZ: {np.round(Ra_monthly, 1)}')
    except Exception as e:
        print(f'   ⚠️  PyAEZ Ra failed ({e}) — using approximation.')
        Ra_monthly = np.full(12, 15.0)
else:
    # Latitude-based Ra approximation for East Africa
    # Ra peaks near equinoxes, dips at solstices — roughly sinusoidal
    _lat_rad = np.radians(abs(lat_c))
    Ra_monthly = 15.0 + 3.0 * np.cos(np.linspace(0, 2*np.pi, 12) + np.pi/6)
    print(f'   ✅ Ra (approximation): {np.round(Ra_monthly, 1)}')

days_per_month = np.array([31,28,31,30,31,30,31,31,30,31,30,31])

# 1D PET (for charts)
pet_monthly_1d = (
    0.0023 * Ra_monthly
    * (tavg_monthly_1d + 17.8)
    * np.sqrt(np.maximum(tmax_monthly_1d - tmin_monthly_1d, 0.1))
)
pet_monthly_mm = pet_monthly_1d * days_per_month

# Spatial PET grid (for maps) — only if we have spatial temperature
if HAS_SPATIAL and tavg_clim is not None:
    pet_clim = np.zeros_like(tavg_clim)  # (12, nrows, ncols)
    for m in range(12):
        _td = np.sqrt(np.maximum(tmax_clim[m] - tmin_clim[m], 0.1))
        pet_clim[m] = 0.0023 * Ra_monthly[m] * (tavg_clim[m] + 17.8) * _td * days_per_month[m]
    print(f'   ✅ Spatial PET grid: {pet_clim.shape}')
else:
    pet_clim = None

water_balance_monthly = precip_monthly_1d - pet_monthly_mm
print(f'\n   PET (mm/month):        {np.round(pet_monthly_mm, 0)}')
print(f'   Water balance (P-PET): {np.round(water_balance_monthly, 0)}')

# ── DERIVED SPATIAL MAPS ──────────────────────────────────────────────────────
# Compute spatial versions of LGP, thermal zone, moisture zone pixel-by-pixel

print('\n🗺️  Computing spatial AEZ maps...')

if HAS_SPATIAL and tavg_clim is not None:
    _nrows_s, _ncols_s = tavg_clim.shape[1], tavg_clim.shape[2]

    # Spatial PET (already computed above)
    _pet_spatial = pet_clim if pet_clim is not None else np.stack([
        0.0023 * Ra_monthly[m] * (tavg_clim[m] + 17.8)
        * np.sqrt(np.maximum(tmax_clim[m] - tmin_clim[m], 0.1))
        * days_per_month[m]
        for m in range(12)
    ], axis=0)

    # Spatial precipitation — regrid to match temperature grid if needed
    if precip_clim is not None and precip_clim.ndim == 3:
        from scipy.interpolate import RegularGridInterpolator
        _plat = next((c for c in precip_ds.coords if 'lat' in c.lower()), None)
        _plon = next((c for c in precip_ds.coords if 'lon' in c.lower()), None)
        _p_lats = precip_ds[_plat].values if _plat else np.linspace(ymin, ymax, precip_clim.shape[1])
        _p_lons = precip_ds[_plon].values if _plon else np.linspace(xmin, xmax, precip_clim.shape[2])

        # Regrid each month to match ERA5 grid
        _prec_on_era5 = np.zeros((12, _nrows_s, _ncols_s))
        for m in range(12):
            try:
                _interp = RegularGridInterpolator(
                    (np.sort(_p_lats), np.sort(_p_lons)),
                    precip_clim[m] if _p_lats[0] < _p_lats[-1] else precip_clim[m][::-1],
                    method='linear', bounds_error=False, fill_value=np.nan
                )
                _grid_lats, _grid_lons = np.meshgrid(map_lats, map_lons, indexing='ij')
                _prec_on_era5[m] = _interp(np.stack([_grid_lats, _grid_lons], axis=-1))
            except Exception:
                _prec_on_era5[m] = precip_monthly_1d[m]  # fallback to scalar
        precip_spatial = _prec_on_era5
    elif precip_clim is not None:
        # 1D — broadcast to spatial
        precip_spatial = precip_monthly_1d[:, np.newaxis, np.newaxis] * np.ones((12, _nrows_s, _ncols_s))
    else:
        precip_spatial = precip_monthly_1d[:, np.newaxis, np.newaxis] * np.ones((12, _nrows_s, _ncols_s))

    # ── SPATIAL LGP MAP ───────────────────────────────────────────────────────
    # LGP = sum of days in months where P >= 0.5 × PET
    _growing_mask = (precip_spatial >= 0.5 * _pet_spatial)  # (12, nrows, ncols)
    lgp_map = (_growing_mask.sum(axis=0) * 30).astype(float)  # days

    # ── SPATIAL THERMAL ZONE MAP ──────────────────────────────────────────────
    _t_min_grid  = tavg_clim.min(axis=0)     # coldest month per pixel
    _t_mean_grid = tavg_clim.mean(axis=0)    # annual mean per pixel
    _cold_months = (tavg_clim < 5).sum(axis=0)

    thermal_map = np.zeros((_nrows_s, _ncols_s), dtype=float)
    thermal_map[(  _t_min_grid >= 20)]                               = 1
    thermal_map[( (_t_min_grid >= 17) & (_t_min_grid < 20) & (_cold_months == 0))] = 2
    thermal_map[( (_t_mean_grid >= 15) & (_cold_months < 3) & (thermal_map == 0))] = 3
    thermal_map[( (_t_mean_grid >= 10) & (_cold_months < 6) & (thermal_map == 0))] = 4
    thermal_map[( (_t_mean_grid >= 5)  & (thermal_map == 0))]        = 5
    thermal_map[( (_t_mean_grid >= 0)  & (thermal_map == 0))]        = 6
    thermal_map[(thermal_map == 0)]                                   = 7

    # ── SPATIAL MOISTURE ZONE MAP ─────────────────────────────────────────────
    moisture_map = np.zeros((_nrows_s, _ncols_s), dtype=float)
    moisture_map[lgp_map < 30]  = 1
    moisture_map[(lgp_map >= 30)  & (lgp_map < 60)]  = 2
    moisture_map[(lgp_map >= 60)  & (lgp_map < 120)] = 3
    moisture_map[(lgp_map >= 120) & (lgp_map < 180)] = 4
    moisture_map[(lgp_map >= 180) & (lgp_map < 270)] = 5
    moisture_map[lgp_map >= 270] = 6

    # ── SPATIAL AEZ COMPOSITE ─────────────────────────────────────────────────
    # AEZ class = thermal_zone * 10 + moisture_zone (unique combo per pixel)
    aez_map = thermal_map * 10 + moisture_map

    print(f'   ✅ LGP map      : min={lgp_map.min():.0f}  max={lgp_map.max():.0f}  mean={lgp_map.mean():.0f} days')
    print(f'   ✅ Thermal map  : classes {np.unique(thermal_map).astype(int).tolist()}')
    print(f'   ✅ Moisture map : zones {np.unique(moisture_map).astype(int).tolist()}')
    HAS_SPATIAL_MAPS = True

else:
    lgp_map = thermal_map = moisture_map = aez_map = None
    HAS_SPATIAL_MAPS = False
    print('   ⚠️  No spatial grid available — maps will be skipped.')
    print('   Charts and country-mean analysis will still work correctly.')


In [ ]:
# @title ### 5) Module I — Climate Regime Analysis {"display-mode":"form"}
# @markdown Runs PyAEZ ClimateRegime on the full spatial grid where available.

# Scalar fallback dictionaries (always used for text summary)
THERMAL_CLASSES = {
    1: 'Tropics, lowland    (Tavg ≥ 20°C all months)',
    2: 'Tropics, subtropics (Tavg 17–20°C, no frost)',
    3: 'Subtropics, warm    (Tavg 10–20°C, < 3 cold months)',
    4: 'Subtropics, cool    (Tavg 5–15°C, 3–6 cold months)',
    5: 'Temperate, moderate (Tavg 0–10°C, frost present)',
    6: 'Temperate, cool/cold (Tavg < 5°C in winter)',
    7: 'Boreal / Arctic',
}
MOISTURE_CLASSES = {
    1: 'Hyper-arid    (LGP < 30 days)',
    2: 'Arid          (LGP 30–60 days)',
    3: 'Semi-arid     (LGP 60–120 days)',
    4: 'Sub-humid     (LGP 120–180 days)',
    5: 'Humid         (LGP 180–270 days)',
    6: 'Per-humid     (LGP > 270 days)',
}

lgp_value      = None
thermal_class  = None
moisture_class = None

# ── Try PyAEZ on spatial grid first ──────────────────────────────────────────
if PYAEZ_AVAILABLE and HAS_SPATIAL_MAPS and tavg_clim is not None:
    try:
        _nrows_s = tavg_clim.shape[1]
        _ncols_s = tavg_clim.shape[2]
        _mask = np.ones((_nrows_s, _ncols_s), dtype=int)

        # PyAEZ expects (12, nrows, ncols)
        _tavg_3d = tavg_clim                        # already (12, nrows, ncols)
        _tmin_3d = tmin_clim
        _tmax_3d = tmax_clim
        _prec_3d = precip_spatial                   # regridded in Cell 5
        _pet_3d  = pet_clim if pet_clim is not None else _pet_spatial

        cr = ClimateRegime.ClimateRegime()
        cr.setStudyAreaMask(_mask, no_data_value=0)
        cr.setLocationTeperature(
            minT_monthly=_tmin_3d,
            maxT_monthly=_tmax_3d,
            meanT_monthly=_tavg_3d
        )
        cr.setMonthlyPrecipitation(_prec_3d)
        cr.setMonthlyPET(_pet_3d)

        cr.getThermalClimate()
        cr.getLGP()
        cr.getMoistureZone()

        # PyAEZ outputs are (nrows, ncols) — overwrite our manual maps
        lgp_map      = cr.lgp.astype(float)
        thermal_map  = cr.thermal_climate.astype(float)
        moisture_map = cr.moisture_zone.astype(float)
        aez_map      = thermal_map * 10 + moisture_map

        # Country-mean scalars from PyAEZ spatial output
        lgp_value      = int(np.nanmean(lgp_map))
        thermal_class  = int(np.nanmedian(thermal_map))
        moisture_class = int(np.nanmedian(moisture_map))

        print('✅ PyAEZ ClimateRegime ran on FULL SPATIAL GRID.')
        print(f'   LGP map      : min={lgp_map.min():.0f}  max={lgp_map.max():.0f}  mean={lgp_map.mean():.0f} days')
        print(f'   Thermal zones: {np.unique(thermal_map).astype(int).tolist()}')
        print(f'   Moisture zones: {np.unique(moisture_map).astype(int).tolist()}')

    except Exception as e:
        print(f'⚠️  PyAEZ spatial run failed: {e}')
        print('   Using manually computed maps from Cell 4 instead.')

# ── Country-mean scalars from our manual maps (fallback or complement) ────────
if lgp_value is None:
    if HAS_SPATIAL_MAPS and lgp_map is not None:
        lgp_value      = int(np.nanmean(lgp_map))
        thermal_class  = int(np.nanmedian(thermal_map))
        moisture_class = int(np.nanmedian(moisture_map))
    else:
        # Pure 1D fallback
        def calc_lgp(p, pet): return int((p >= 0.5 * pet).sum() * 30)
        def classify_thermal(tavg):
            if tavg.min() >= 20: return 1
            if tavg.min() >= 17: return 2
            if tavg.mean() >= 15: return 3
            if tavg.mean() >= 10: return 4
            if tavg.mean() >= 5: return 5
            if tavg.mean() >= 0: return 6
            return 7
        def classify_moisture(lgp):
            for thr, cls in [(30,1),(60,2),(120,3),(180,4),(270,5)]:
                if lgp < thr: return cls
            return 6
        lgp_value      = calc_lgp(precip_monthly_1d, pet_monthly_mm)
        thermal_class  = classify_thermal(tavg_monthly_1d)
        moisture_class = classify_moisture(lgp_value)

tavg_annual  = tavg_monthly_1d.mean()
precip_annual = precip_monthly_1d.sum()

print('\n' + '═'*60)
print(f'  CLIMATE REGIME RESULTS — {COUNTRY}')
print('═'*60)
print(f'  Thermal class   : {thermal_class} — {THERMAL_CLASSES.get(thermal_class,"Unknown")}')
print(f'  LGP (mean)      : {lgp_value} days/year')
print(f'  Moisture zone   : {moisture_class} — {MOISTURE_CLASSES.get(moisture_class,"Unknown")}')
print(f'  Annual Tavg     : {tavg_annual:.1f}°C')
print(f'  Annual Precip   : {precip_annual:.0f} mm')
print(f'  Annual PET      : {pet_monthly_mm.sum():.0f} mm')
print(f'  Aridity index   : {precip_annual/pet_monthly_mm.sum():.2f} (P/PET)')
print(f'  Spatial maps    : {"✅ available" if HAS_SPATIAL_MAPS else "⚠️  not available (1D mode)"}')
print('═'*60)


In [ ]:
# @title ### 5b) Geospatial AEZ Maps {"display-mode":"form"}
# @markdown Renders spatial maps of LGP, Thermal Zone, Moisture Zone, and Annual Precipitation
# @markdown across the selected country. Requires the spatial grid from Cell 4.

if not HAS_SPATIAL_MAPS or lgp_map is None:
    print('⚠️  Spatial maps not available.')
    print('   This happens when ERA5 temperature was loaded as a 1D country-mean series')
    print('   rather than a full spatial grid.')
    print('   Re-run Cell 3 — it will pull the spatial grid from GEE and cache it.')
else:
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    import matplotlib.patches as mpatches

    fig = plt.figure(figsize=(20, 16))
    fig.suptitle(
        f'Focus Area 4 — Agro-Ecological Zone Maps\n{COUNTRY}  |  {LT_START}–{LT_END}',
        fontsize=15, fontweight='bold', y=0.99
    )

    _extent = [xmin, xmax, ymin, ymax]

    # ── MAP 1: Length of Growing Period ───────────────────────────────────────
    ax1 = fig.add_subplot(2, 2, 1)
    _lgp_cmap = plt.cm.RdYlGn
    _lgp_norm = mcolors.Normalize(vmin=0, vmax=365)
    im1 = ax1.imshow(
        lgp_map, extent=_extent, cmap=_lgp_cmap, norm=_lgp_norm,
        origin='upper', aspect='auto', interpolation='bilinear'
    )
    plt.colorbar(im1, ax=ax1, label='Days / year', shrink=0.8, pad=0.02)
    ax1.set_title(f'Length of Growing Period (LGP)\nMean = {lgp_map.mean():.0f} days/yr', fontsize=11)
    ax1.set_xlabel('Longitude'); ax1.set_ylabel('Latitude')
    ax1.set_xlim(xmin, xmax); ax1.set_ylim(ymin, ymax)
    ax1.grid(True, alpha=0.3, color='white', linewidth=0.5)
    # LGP threshold lines as contours
    try:
        _cs = ax1.contour(
            np.linspace(xmin, xmax, lgp_map.shape[1]),
            np.linspace(ymin, ymax, lgp_map.shape[0]),
            lgp_map,
            levels=[60, 120, 180, 270],
            colors='white', linewidths=0.8, alpha=0.7
        )
        ax1.clabel(_cs, fmt='%d d', fontsize=7, colors='white')
    except Exception:
        pass

    # ── MAP 2: Thermal Zone ────────────────────────────────────────────────────
    ax2 = fig.add_subplot(2, 2, 2)
    _t_colors = ['#d73027','#fc8d59','#fee08b','#d9ef8b','#91cf60','#1a9850','#4575b4']
    _t_cmap   = mcolors.ListedColormap(_t_colors[:int(thermal_map.max())])
    _t_norm   = mcolors.BoundaryNorm(np.arange(0.5, thermal_map.max()+1.5), _t_cmap.N)
    im2 = ax2.imshow(
        thermal_map, extent=_extent, cmap=_t_cmap, norm=_t_norm,
        origin='upper', aspect='auto', interpolation='nearest'
    )
    _t_labels_short = {
        1:'Tropics lowland', 2:'Tropics/subtropics',
        3:'Subtropics warm', 4:'Subtropics cool',
        5:'Temperate', 6:'Temperate cool', 7:'Boreal'
    }
    _t_patches = [
        mpatches.Patch(color=_t_colors[int(v)-1], label=f'Class {int(v)}: {_t_labels_short.get(int(v),"?")}')
        for v in np.unique(thermal_map) if not np.isnan(v)
    ]
    ax2.legend(handles=_t_patches, loc='lower left', fontsize=7, framealpha=0.8)
    ax2.set_title('Thermal Climate Classification', fontsize=11)
    ax2.set_xlabel('Longitude'); ax2.set_ylabel('Latitude')
    ax2.set_xlim(xmin, xmax); ax2.set_ylim(ymin, ymax)
    ax2.grid(True, alpha=0.3, color='white', linewidth=0.5)

    # ── MAP 3: Moisture Zone ───────────────────────────────────────────────────
    ax3 = fig.add_subplot(2, 2, 3)
    _m_colors = ['#d73027','#f46d43','#fdae61','#fee090','#74add1','#313695']
    _m_cmap   = mcolors.ListedColormap(_m_colors[:int(moisture_map.max())])
    _m_norm   = mcolors.BoundaryNorm(np.arange(0.5, moisture_map.max()+1.5), _m_cmap.N)
    im3 = ax3.imshow(
        moisture_map, extent=_extent, cmap=_m_cmap, norm=_m_norm,
        origin='upper', aspect='auto', interpolation='nearest'
    )
    _m_labels_short = {
        1:'Hyper-arid (<30d)', 2:'Arid (30–60d)',
        3:'Semi-arid (60–120d)', 4:'Sub-humid (120–180d)',
        5:'Humid (180–270d)', 6:'Per-humid (>270d)'
    }
    _m_patches = [
        mpatches.Patch(color=_m_colors[int(v)-1], label=_m_labels_short.get(int(v),'?'))
        for v in np.unique(moisture_map) if not np.isnan(v)
    ]
    ax3.legend(handles=_m_patches, loc='lower left', fontsize=7, framealpha=0.8)
    ax3.set_title('Moisture Zone Classification', fontsize=11)
    ax3.set_xlabel('Longitude'); ax3.set_ylabel('Latitude')
    ax3.set_xlim(xmin, xmax); ax3.set_ylim(ymin, ymax)
    ax3.grid(True, alpha=0.3, color='white', linewidth=0.5)

    # ── MAP 4: Annual Precipitation ────────────────────────────────────────────
    ax4 = fig.add_subplot(2, 2, 4)
    if precip_clim is not None and precip_clim.ndim == 3:
        _ann_precip = precip_clim.sum(axis=0)
    elif precip_clim is not None and precip_clim.ndim == 1:
        _ann_precip = np.full(lgp_map.shape, precip_clim.sum())
    else:
        _ann_precip = np.full(lgp_map.shape, precip_monthly_1d.sum())

    _p_cmap = plt.cm.YlGnBu
    _p_norm = mcolors.Normalize(vmin=0, vmax=max(_ann_precip.max(), 1000))
    im4 = ax4.imshow(
        _ann_precip, extent=_extent, cmap=_p_cmap, norm=_p_norm,
        origin='upper', aspect='auto', interpolation='bilinear'
    )
    plt.colorbar(im4, ax=ax4, label='mm / year', shrink=0.8, pad=0.02)
    ax4.set_title(f'Annual Precipitation ({PRECIP_SOURCE})\nMean = {precip_monthly_1d.sum():.0f} mm/yr', fontsize=11)
    ax4.set_xlabel('Longitude'); ax4.set_ylabel('Latitude')
    ax4.set_xlim(xmin, xmax); ax4.set_ylim(ymin, ymax)
    ax4.grid(True, alpha=0.3, color='white', linewidth=0.5)

    plt.tight_layout(rect=[0, 0, 1, 0.97])

    _maps_path = os.path.join(RESULTS_DIR, f'FA4_AEZ_Maps_{COUNTRY}.png')
    plt.savefig(_maps_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Maps saved to: {_maps_path}')
    print()
    print('MAP INTERPRETATION:')
    print(f'  LGP range    : {lgp_map.min():.0f} – {lgp_map.max():.0f} days/year')
    print(f'  Thermal zones: {sorted(set(int(v) for v in np.unique(thermal_map) if not np.isnan(v)))}')
    print(f'  Moisture zones: {sorted(set(int(v) for v in np.unique(moisture_map) if not np.isnan(v)))}')
    print(f'  Annual precip range: {_ann_precip.min():.0f} – {_ann_precip.max():.0f} mm/year')


In [ ]:
# @title ### 6) Crop suitability assessment {"display-mode":"form"}
# @markdown Evaluates which crops are thermally and moisture-suitable for the country
# @markdown using FAO AEZ thermal and water requirements.
# @markdown Crops relevant to East Africa: Maize, Sorghum, Wheat, Millet, Rice, Cassava.

# ── CROP REQUIREMENTS (FAO AEZ) ───────────────────────────────────────────────
# Source: FAO GAEZ v4 crop parameter tables
CROPS = {
    'Maize': {
        'tavg_min': 10.0, 'tavg_max': 35.0,
        'tmin_abs': 5.0,  'tmax_abs': 40.0,
        'lgp_min': 90,    'lgp_opt': 150,
        'precip_min': 400, 'precip_opt': 700,
        'description': 'Most important food crop in East Africa.'
    },
    'Sorghum': {
        'tavg_min': 12.0, 'tavg_max': 37.0,
        'tmin_abs': 6.0,  'tmax_abs': 42.0,
        'lgp_min': 75,    'lgp_opt': 120,
        'precip_min': 250, 'precip_opt': 500,
        'description': 'Drought-tolerant. Suited to semi-arid zones.'
    },
    'Wheat': {
        'tavg_min': 5.0,  'tavg_max': 25.0,
        'tmin_abs': 0.0,  'tmax_abs': 32.0,
        'lgp_min': 90,    'lgp_opt': 150,
        'precip_min': 300, 'precip_opt': 600,
        'description': 'Higher elevations (Ethiopia highlands).'
    },
    'Millet (Pearl)': {
        'tavg_min': 14.0, 'tavg_max': 38.0,
        'tmin_abs': 8.0,  'tmax_abs': 44.0,
        'lgp_min': 60,    'lgp_opt': 100,
        'precip_min': 200, 'precip_opt': 400,
        'description': 'Most drought-tolerant cereal. Suited to arid margins.'
    },
    'Rice (Upland)': {
        'tavg_min': 18.0, 'tavg_max': 35.0,
        'tmin_abs': 12.0, 'tmax_abs': 38.0,
        'lgp_min': 90,    'lgp_opt': 150,
        'precip_min': 800, 'precip_opt': 1200,
        'description': 'Humid lowlands and irrigated areas.'
    },
    'Cassava': {
        'tavg_min': 18.0, 'tavg_max': 35.0,
        'tmin_abs': 10.0, 'tmax_abs': 38.0,
        'lgp_min': 120,   'lgp_opt': 240,
        'precip_min': 500, 'precip_opt': 1000,
        'description': 'Resilient root crop. Tolerates poor soils.'
    },
}

# ── ASSESS SUITABILITY ────────────────────────────────────────────────────────
tavg_annual   = tavg_monthly_1d.mean()
tmin_annual   = tmin_monthly_1d.min()
tmax_annual   = tmax_monthly_1d.max()
precip_annual = precip_monthly_1d.sum()

results = []

for crop, req in CROPS.items():
    # Thermal suitability
    thermal_ok = (
        req['tavg_min'] <= tavg_annual <= req['tavg_max']
        and tmin_annual >= req['tmin_abs']
        and tmax_annual <= req['tmax_abs'] + 3  # small tolerance
    )

    # LGP suitability
    lgp_ok  = lgp_value >= req['lgp_min']
    lgp_opt = lgp_value >= req['lgp_opt']

    # Water suitability
    water_ok  = precip_annual >= req['precip_min']
    water_opt = precip_annual >= req['precip_opt']

    # Overall suitability
    if thermal_ok and lgp_ok and water_ok:
        if lgp_opt and water_opt:
            suitability = 'High'
            score = 3
        else:
            suitability = 'Moderate'
            score = 2
    elif thermal_ok and (lgp_ok or water_ok):
        suitability = 'Marginal'
        score = 1
    else:
        suitability = 'Not suitable'
        score = 0

    results.append({
        'Crop':         crop,
        'Suitability':  suitability,
        'Score':        score,
        'Thermal OK':   '✅' if thermal_ok else '❌',
        'LGP OK':       '✅' if lgp_ok else '❌',
        'Water OK':     '✅' if water_ok else '❌',
        'Description':  req['description'],
    })

df_suitability = pd.DataFrame(results).sort_values('Score', ascending=False)

print(f'\nCROP SUITABILITY ASSESSMENT — {COUNTRY}')
print(f'  Tavg={tavg_annual:.1f}°C  Tmin={tmin_annual:.1f}°C  Tmax={tmax_annual:.1f}°C')
print(f'  LGP={lgp_value} days  Precip={precip_annual:.0f}mm')
print()
from IPython.display import display
display(df_suitability[['Crop','Suitability','Thermal OK','LGP OK','Water OK','Description']])

In [ ]:
# @title ### 7) Results visualisation {"display-mode":"form"}
# @markdown Plots the monthly climate profile, water balance, and crop suitability summary.

fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    f'Focus Area 4 — Agro-Ecological Zoning\n{COUNTRY}  |  {LT_START}–{LT_END}',
    fontsize=14, fontweight='bold', y=0.98
)

# ── Panel 1: Temperature profile ──────────────────────────────────────────────
ax1 = fig.add_subplot(3, 3, 1)
ax1.fill_between(range(12), tmin_monthly_1d, tmax_monthly_1d, alpha=0.2, color='tomato', label='Tmin–Tmax range')
ax1.plot(range(12), tavg_monthly_1d, 'o-', color='tomato', lw=2, ms=5, label='Tavg')
ax1.axhline(10, color='steelblue', ls='--', lw=0.8, label='10°C (crop base)')
ax1.axhline(0,  color='black',     ls='--', lw=0.8, label='0°C (frost)')
ax1.set_xticks(range(12)); ax1.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('°C'); ax1.set_title('Monthly Temperature')
ax1.legend(fontsize=7); ax1.grid(True, alpha=0.3)

# ── Panel 2: Precipitation vs PET ─────────────────────────────────────────────
ax2 = fig.add_subplot(3, 3, 2)
ax2.bar(range(12), precip_monthly_1d, color='steelblue', alpha=0.8, label='Precipitation')
ax2.plot(range(12), pet_monthly_mm, 'o-', color='tomato', lw=2, ms=5, label='PET (Hargreaves)')
ax2.set_xticks(range(12)); ax2.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('mm'); ax2.set_title('Precipitation vs PET')
ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3, axis='y')

# ── Panel 3: Water balance ────────────────────────────────────────────────────
ax3 = fig.add_subplot(3, 3, 3)
colors_wb = ['steelblue' if v >= 0 else 'tomato' for v in water_balance_monthly]
ax3.bar(range(12), water_balance_monthly, color=colors_wb, alpha=0.85)
ax3.axhline(0, color='black', lw=1)
ax3.set_xticks(range(12)); ax3.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('mm'); ax3.set_title('Water Balance (P − PET)')
ax3.grid(True, alpha=0.3, axis='y')

# ── Panel 4: Crop suitability bar chart ───────────────────────────────────────
ax4 = fig.add_subplot(3, 3, 4)
_suit_colors = {'High':'#2e7d52','Moderate':'#c87c1a','Marginal':'#d4a017','Not suitable':'#b5341a'}
ax4.barh(
    df_suitability['Crop'],
    df_suitability['Score'],
    color=[_suit_colors[s] for s in df_suitability['Suitability']],
    alpha=0.85
)
ax4.set_xlim(0, 3.5)
ax4.set_xticks([0,1,2,3])
ax4.set_xticklabels(['Not\nSuitable','Marginal','Moderate','High'], fontsize=8)
ax4.set_title('Crop Suitability Score')
ax4.grid(True, alpha=0.3, axis='x')

# ── Panel 5: AEZ summary text card ────────────────────────────────────────────
ax5 = fig.add_subplot(3, 3, 5)
ax5.axis('off')
summary_text = (
    f"AEZ SUMMARY\n"
    f"{'─'*28}\n"
    f"Country       : {COUNTRY}\n"
    f"Thermal class : {thermal_class}\n"
    f"  {THERMAL_CLASSES.get(thermal_class,'?')}\n\n"
    f"Moisture zone : {moisture_class}\n"
    f"  {MOISTURE_CLASSES.get(moisture_class,'?')}\n\n"
    f"LGP           : {lgp_value} days/yr\n"
    f"Annual Tavg   : {tavg_annual:.1f}°C\n"
    f"Annual Precip : {precip_annual:.0f} mm\n"
    f"Annual PET    : {pet_monthly_mm.sum():.0f} mm\n"
    f"Aridity (P/PET): {precip_annual/pet_monthly_mm.sum():.2f}\n\n"
    f"Precip source : {PRECIP_SOURCE}\n"
    f"Period        : {LT_START}–{LT_END}"
)
ax5.text(0.05, 0.95, summary_text, transform=ax5.transAxes,
         va='top', fontsize=9, fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#f4f6fb', alpha=0.8))

# ── Panel 6: Growing period moisture indicator ─────────────────────────────────
ax6 = fig.add_subplot(3, 3, 6)
growing = (precip_monthly_1d >= 0.5 * pet_monthly_mm).astype(int)
ax6.bar(range(12), growing, color=['#2e7d52' if g else '#e8ddd0' for g in growing], alpha=0.85)
ax6.set_xticks(range(12)); ax6.set_xticklabels(MONTHS, rotation=45, ha='right', fontsize=8)
ax6.set_yticks([0, 1]); ax6.set_yticklabels(['Dry', 'Growing'])
ax6.set_title(f'Monthly Growing Conditions\n(LGP ≈ {lgp_value} days/year)')
ax6.grid(True, alpha=0.3, axis='y')

# ── Panel 7–9: Long-term temperature trend ────────────────────────────────────
if era5_lt is not None:
    ax7 = fig.add_subplot(3, 1, 3)
    _var0 = list(era5_lt.data_vars)[0]
    _da   = era5_lt[_var0]
    _sp   = [d for d in _da.dims if d != 'time']
    _vals = _da.mean(dim=_sp).values if _sp else _da.values
    if float(np.nanmean(_vals)) > 100:
        _vals = _vals - 273.15
    _times = pd.DatetimeIndex([pd.Timestamp(str(t)[:10]) for t in era5_lt.time.values])
    ax7.plot(_times, _vals, color='tomato', lw=1.2, alpha=0.8)
    # 12-month rolling mean
    _rolling = pd.Series(_vals, index=_times).rolling(12, center=True).mean()
    ax7.plot(_rolling.index, _rolling.values, color='darkred', lw=2, label='12-month mean')
    ax7.set_ylabel('Temperature (°C)')
    ax7.set_title(f'Long-term ERA5 Temperature — {COUNTRY}  ({LT_START}–{LT_END})')
    ax7.legend(fontsize=8); ax7.grid(True, alpha=0.3)

plt.tight_layout()

# Save
_plot_path = os.path.join(RESULTS_DIR, f'FA4_AEZ_{COUNTRY}.png')
plt.savefig(_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plot saved to: {_plot_path}')

# Also save the suitability table
_csv_path = os.path.join(RESULTS_DIR, f'FA4_crop_suitability_{COUNTRY}.csv')
df_suitability.to_csv(_csv_path, index=False)
print(f'✅ Suitability table saved to: {_csv_path}')

In [ ]:
# @title ### 8) Cache results to Shared Drive {"display-mode":"form"}
# @markdown Uploads the AEZ outputs to the Shared Drive so other participants
# @markdown running the same country can access pre-computed results.

from google.colab import files

print(f'☁️  Caching FA4 results to Shared Drive/{COUNTRY}...')

_outputs = [
    (f'FA4_AEZ_{COUNTRY}.png',              os.path.join(RESULTS_DIR, f'FA4_AEZ_{COUNTRY}.png')),
    (f'FA4_crop_suitability_{COUNTRY}.csv', os.path.join(RESULTS_DIR, f'FA4_crop_suitability_{COUNTRY}.csv')),
]

for drive_name, local_path in _outputs:
    if os.path.exists(local_path):
        cache_to_drive(local_path, drive_name)
    else:
        print(f'   ⚠️  Skipping {drive_name} — file not found (run Cell 7 first)')

print('\n📥 Downloading results to your local machine...')
for _, local_path in _outputs:
    if os.path.exists(local_path):
        files.download(local_path)

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print('\n' + '═'*60)
print(f'  FOCUS AREA 4 COMPLETE — {COUNTRY}')
print('═'*60)
print(f'  Thermal class  : {thermal_class} — {THERMAL_CLASSES.get(thermal_class,"?")[:35]}')
print(f'  Moisture zone  : {moisture_class} — {MOISTURE_CLASSES.get(moisture_class,"?")[:35]}')
print(f'  LGP            : {lgp_value} days/year')
print()

_high  = df_suitability[df_suitability['Suitability']=='High']['Crop'].tolist()
_mod   = df_suitability[df_suitability['Suitability']=='Moderate']['Crop'].tolist()
_marg  = df_suitability[df_suitability['Suitability']=='Marginal']['Crop'].tolist()
_none  = df_suitability[df_suitability['Suitability']=='Not suitable']['Crop'].tolist()

if _high:  print(f'  ✅ High suitability     : {", ".join(_high)}')
if _mod:   print(f'  🟡 Moderate suitability : {", ".join(_mod)}')
if _marg:  print(f'  🟠 Marginal suitability : {", ".join(_marg)}')
if _none:  print(f'  ❌ Not suitable         : {", ".join(_none)}')
print('═'*60)